In [1]:
# 07_recourse.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 07. Recourse Feasibility Axis
#
# Add the third framework axis to the GGI: recourse feasibility.
#
# Definition (oriented so higher = better):
#   s_REC = (# adversely-predicted individuals with a FEASIBLE counterfactual)
#           / (# adversely-predicted individuals)
# "Adverse" = predicted high-risk (label 1). A counterfactual is feasible only if it
# flips the decision by changing MUTABLE features (immutable attributes such as age or
# sex are frozen). Recourse is generated with DiCE on the raw (pre-encoding) feature space.
#
# Scope: recourse is computed only for datasets whose features are semantically
# interpretable (mutable vs immutable is known). Porto Seguro uses anonymized proxy
# variables, so recourse is undefined there and reported as N/A.

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

import dice_ml
from dice_ml import Dice

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
N_ADVERSE = 100        # number of adversely-predicted individuals to test recourse for
N_CF = 3               # counterfactuals requested per individual
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Recourse specification per dataset:
#   loader   -> returns a raw feature DataFrame + target Series
#   immutable-> features frozen during counterfactual search
#   continuous-> continuous feature names (rest treated as categorical by DiCE)
#
# We rebuild raw frames from the parquet saved in notebook 01 (before encoding).

def load_raw(key):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}.parquet"))

# German Credit
def spec_german():
    df = load_raw("german")
    keep_drop = ["personal_status_sex", "sex", "age_group"]  # keep 'age' as continuous
    df = df.drop(columns=[c for c in keep_drop if c in df.columns], errors="ignore")
    target = "target"
    immutable = ["age", "foreign_worker"]
    continuous = ["duration", "credit_amount", "age", "installment_rate",
                  "residence_since", "existing_credits", "num_dependents"]
    continuous = [c for c in continuous if c in df.columns]
    return df, target, immutable, continuous

# Credit Card Approval
def spec_credit_card():
    df = load_raw("credit_card_approval")
    drop = ["ID", "sex", "target_30dpd", "target_60dpd", "domain"]
    df = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    target = "target"
    immutable = ["CODE_GENDER", "DAYS_BIRTH", "FLAG_OWN_REALTY", "FLAG_OWN_CAR"]
    immutable = [c for c in immutable if c in df.columns]
    continuous = ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "DAYS_BIRTH",
                  "DAYS_EMPLOYED", "CNT_FAM_MEMBERS"]
    continuous = [c for c in continuous if c in df.columns]
    return df, target, immutable, continuous

# Medical Cost
def spec_medical():
    df = load_raw("insurance")
    drop = ["age_group", "smoker_flag", "domain", "charges"]  # charges leaks target
    df = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    target = "target"
    immutable = ["age", "sex"]
    continuous = ["age", "bmi", "children"]
    continuous = [c for c in continuous if c in df.columns]
    return df, target, immutable, continuous

SPECS = {
    "German Credit": spec_german,
    "Credit Card Approval": spec_credit_card,
    "Medical Cost": spec_medical,
}
UNDEFINED = ["Porto Seguro"]  # anonymized proxies -> recourse undefined

# %%
# Build a DiCE-compatible sklearn pipeline (encoding lives inside the model) and
# compute s_REC = fraction of adverse predictions with a feasible counterfactual.
def compute_srec(df, target, immutable, continuous,
                 n_adverse=N_ADVERSE, n_cf=N_CF, seed=RANDOM_STATE):
    y = df[target].astype(int)
    X = df.drop(columns=[target])
    cont = [c for c in continuous if c in X.columns]
    cat = [c for c in X.columns if c not in cont]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )

    pre = ColumnTransformer([
        ("num", StandardScaler(), cont),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    clf = Pipeline([
        ("pre", pre),
        ("model", RandomForestClassifier(
            n_estimators=300, max_depth=8, class_weight="balanced",
            random_state=seed, n_jobs=-1)),
    ])
    clf.fit(X_train, y_train)

    # Identify adversely-predicted individuals in the test set (predicted high-risk = 1)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    adverse_idx = X_test.index[pred == 1]
    if len(adverse_idx) == 0:
        return np.nan, 0, 0
    adverse_idx = adverse_idx[:n_adverse]
    query = X_test.loc[adverse_idx].copy()

    # DiCE setup: mutable features are all non-immutable columns
    features_to_vary = [c for c in X.columns if c not in immutable]

    train_df = X_train.copy()
    train_df[target] = y_train.values
    d = dice_ml.Data(
        dataframe=train_df,
        continuous_features=cont,
        outcome_name=target,
    )
    m = dice_ml.Model(model=clf, backend="sklearn")
    exp = Dice(d, m, method="random")

    feasible = 0
    tested = 0
    for i in range(len(query)):
        instance = query.iloc[[i]]
        tested += 1
        try:
            cf = exp.generate_counterfactuals(
                instance, total_CFs=n_cf, desired_class=0,
                features_to_vary=features_to_vary,
            )
            cf_df = cf.cf_examples_list[0].final_cfs_df
            if cf_df is not None and len(cf_df) > 0:
                feasible += 1
        except Exception:
            # No feasible counterfactual found within mutable-feature constraints
            pass

    s_rec = feasible / tested if tested else np.nan
    return s_rec, feasible, tested

# %%
# Run recourse computation for each in-scope dataset
rec_rows = []
for name, spec_fn in SPECS.items():
    df, target, immutable, continuous = spec_fn()
    s_rec, feasible, tested = compute_srec(df, target, immutable, continuous)
    rec_rows.append({
        "dataset": name, "s_REC": round(s_rec, 4) if np.isfinite(s_rec) else np.nan,
        "feasible": feasible, "tested": tested,
        "immutable": ", ".join(immutable),
    })
    print(f"[{name:>22}] s_REC={s_rec:.3f}  ({feasible}/{tested} adverse cases)")

for name in UNDEFINED:
    rec_rows.append({"dataset": name, "s_REC": np.nan,
                     "feasible": 0, "tested": 0, "immutable": "N/A (anonymized proxies)"})
    print(f"[{name:>22}] s_REC=N/A (recourse undefined for anonymized features)")

rec_df = pd.DataFrame(rec_rows)
print(rec_df.to_string(index=False))
rec_df.to_csv(os.path.join(TAB_DIR, "07_recourse_scores.csv"), index=False)

# %%
# Integrate s_REC into the GGI as a fourth score axis (fifth component with performance).
# Add a recourse threshold to each regulation's strictness gradient (modeled scenario).
REC_THRESHOLDS = {"KR_AI_Basic_Act": 0.60, "EU_AI_Act": 0.70, "NAIC": 0.65}

comp_scores = pd.read_csv(os.path.join(TAB_DIR, "04_component_scores.csv"))
comp_scores = comp_scores.merge(rec_df[["dataset", "s_REC"]], on="dataset", how="left")

def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

# Recompute an EXTENDED GGI that includes recourse where available.
BASE_REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}

ext_rows = []
for _, r in comp_scores.iterrows():
    for reg, tau in BASE_REGULATIONS.items():
        gaps = [
            component_gap(r["s_PERF"], tau["s_PERF"]),
            component_gap(r["s_DI"], tau["s_DI"]),
            component_gap(r["s_SP"], tau["s_SP"]),
            component_gap(r["s_EXPL"], tau["s_EXPL"]),
        ]
        g_rec = component_gap(r["s_REC"], REC_THRESHOLDS[reg]) if np.isfinite(r["s_REC"]) else np.nan
        if np.isfinite(g_rec):
            gaps.append(g_rec)
        valid = [g for g in gaps if np.isfinite(g)]
        ggi_ext = float(np.sqrt(np.sum(np.square(valid)))) if valid else np.nan
        ext_rows.append({
            "domain": r["domain"], "dataset": r["dataset"], "regulation": reg,
            "s_REC": r["s_REC"], "g_REC": g_rec, "GGI_extended": round(ggi_ext, 4),
            "recourse_included": np.isfinite(g_rec),
        })

ext_df = pd.DataFrame(ext_rows)
print(ext_df.to_string(index=False))
ext_df.to_csv(os.path.join(TAB_DIR, "07_ggi_extended.csv"), index=False)

# %%
# Comparison: GGI with vs without the recourse axis (KR baseline), in-scope datasets only
base = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"))
base_kr = base[base["regulation"] == "KR_AI_Basic_Act"][["dataset", "GGI"]].rename(
    columns={"GGI": "GGI_4axis"})
ext_kr = ext_df[ext_df["regulation"] == "KR_AI_Basic_Act"][
    ["dataset", "GGI_extended", "recourse_included"]]
compare = base_kr.merge(ext_kr, on="dataset")
print(compare.to_string(index=False))
compare.to_csv(os.path.join(TAB_DIR, "07_ggi_recourse_comparison.csv"), index=False)

100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


[         German Credit] s_REC=1.000  (74/74 adverse cases)


100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


[  Credit Card Approval] s_REC=0.840  (84/100 adverse cases)


100%|██████████| 1/1 [00:01<00:00,  1.06s/it]

[          Medical Cost] s_REC=1.000  (61/61 adverse cases)
[          Porto Seguro] s_REC=N/A (recourse undefined for anonymized features)
             dataset  s_REC  feasible  tested                                              immutable
       German Credit   1.00        74      74                                    age, foreign_worker
Credit Card Approval   0.84        84     100 CODE_GENDER, DAYS_BIRTH, FLAG_OWN_REALTY, FLAG_OWN_CAR
        Medical Cost   1.00        61      61                                               age, sex
        Porto Seguro    NaN         0       0                               N/A (anonymized proxies)
   domain              dataset      regulation  s_REC  g_REC  GGI_extended  recourse_included
   credit        German Credit KR_AI_Basic_Act   1.00    0.0        0.6818               True
   credit        German Credit       EU_AI_Act   1.00    0.0        0.7350               True
   credit        German Credit            NAIC   1.00    0.0        0.712